# AgroVision AI - Combined Plant Disease Training

This Colab notebook trains a combined model using your existing crop disease dataset plus a new medicinal plant dataset.

**Recommended model:** EfficientNetV2B0 by default. It is accurate, fast enough for Colab, and deployable. If you have Colab Pro/GPU memory, switch to EfficientNetV2B3 for better accuracy.

The notebook first stages the dataset into Colab local storage under `/content`, then trains from that local copy. This is faster and more stable than reading images directly from Google Drive during every epoch.

Expected dataset format:

```text
dataset_root/
  Apple___Apple_scab/
  Tomato___Early_blight/
  MedicinalPlant___DiseaseName/
  MedicinalPlant___healthy/
  Background_without_leaves/
```

The final output is compatible with the current FastAPI backend:

- `EfficientNetV2_combined.h5`
- `class_names_combined.json`


In [ ]:
# Colab setup
from google.colab import drive
drive.mount('/content/drive')

import os, json, shutil, random, zipfile
from pathlib import Path
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

SEED = 42
tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

## 1. Stage Dataset To Colab Local Storage

Put your combined dataset in Google Drive as either a folder of class folders or a ZIP file. This cell copies/extracts it into Colab local storage first. Training then uses `/content/agrovision_local_dataset` directly, which is usually much faster than streaming thousands of images from Drive.

In [ ]:
# Source dataset in Google Drive. This can be either:
# 1) a folder containing class folders, or
# 2) a .zip file containing the dataset folder/class folders.
DRIVE_DATASET_SOURCE = Path('/content/drive/MyDrive/AgroVision/dataset_combined')

# Local Colab training copy. The model will train from this path, not Drive.
LOCAL_DATASET_ROOT = Path('/content/agrovision_local_dataset')
RESET_LOCAL_DATASET = False  # set True if you changed the Drive dataset and want a fresh local copy

# Output folder stays in Drive so trained models are saved after the Colab session ends.
OUTPUT_DIR = Path('/content/drive/MyDrive/AgroVision/trained_models')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

image_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def has_image_files(path):
    return any(f.suffix.lower() in image_exts for f in path.rglob('*') if f.is_file())

def looks_like_dataset_root(path):
    class_dirs = [p for p in path.iterdir() if p.is_dir()]
    image_class_dirs = [p for p in class_dirs if has_image_files(p)]
    return len(image_class_dirs) >= 2

def find_dataset_root(path):
    candidates = [path] + [p for p in path.rglob('*') if p.is_dir()]
    for candidate in candidates:
        if looks_like_dataset_root(candidate):
            return candidate
    raise FileNotFoundError(f'Could not find class folders with images inside: {path}')

assert DRIVE_DATASET_SOURCE.exists(), f'Drive dataset source not found: {DRIVE_DATASET_SOURCE}'

if RESET_LOCAL_DATASET and LOCAL_DATASET_ROOT.exists():
    shutil.rmtree(LOCAL_DATASET_ROOT)

if not LOCAL_DATASET_ROOT.exists() or not looks_like_dataset_root(LOCAL_DATASET_ROOT):
    if LOCAL_DATASET_ROOT.exists():
        shutil.rmtree(LOCAL_DATASET_ROOT)

    if DRIVE_DATASET_SOURCE.is_file() and DRIVE_DATASET_SOURCE.suffix.lower() == '.zip':
        extract_dir = Path('/content/agrovision_dataset_extract')
        if extract_dir.exists():
            shutil.rmtree(extract_dir)
        extract_dir.mkdir(parents=True, exist_ok=True)
        print('Extracting dataset zip to Colab local storage...')
        with zipfile.ZipFile(DRIVE_DATASET_SOURCE, 'r') as zf:
            zf.extractall(extract_dir)
        detected_root = find_dataset_root(extract_dir)
        shutil.copytree(detected_root, LOCAL_DATASET_ROOT)
    else:
        detected_root = find_dataset_root(DRIVE_DATASET_SOURCE)
        print('Copying dataset folder to Colab local storage...')
        shutil.copytree(detected_root, LOCAL_DATASET_ROOT)
else:
    print('Using existing local Colab dataset copy.')

DATASET_ROOT = LOCAL_DATASET_ROOT
assert DATASET_ROOT.exists(), f'Local dataset folder not found: {DATASET_ROOT}'

class_dirs = sorted([p for p in DATASET_ROOT.iterdir() if p.is_dir()])
print('Training from local dataset:', DATASET_ROOT)
print('Number of classes:', len(class_dirs))
print('First 10 classes:', [p.name for p in class_dirs[:10]])
print('Output folder:', OUTPUT_DIR)

## 2. Dataset Quality Check\n
\n
This checks class counts. Try to avoid extreme imbalance. If one class has 3000 images and another has 80, training can become biased.

In [ ]:
image_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
counts = {}
for class_dir in class_dirs:
    counts[class_dir.name] = sum(1 for f in class_dir.rglob('*') if f.suffix.lower() in image_exts)

for name, count in sorted(counts.items(), key=lambda x: x[1])[:15]:
    print(f'{name}: {count}')

print('Total images:', sum(counts.values()))
print('Smallest class:', min(counts.items(), key=lambda x: x[1]))
print('Largest class:', max(counts.items(), key=lambda x: x[1]))

## 3. Training Settings\n
\n
Use `efficientnetv2b0` first. If validation accuracy is not enough and Colab memory allows, try `efficientnetv2b3`.

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
MODEL_NAME = 'efficientnetv2b0'  # options: efficientnetv2b0, efficientnetv2b3
INITIAL_EPOCHS = 12
FINE_TUNE_EPOCHS = 12
VALIDATION_SPLIT = 0.2

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_ROOT,
    validation_split=VALIDATION_SPLIT,
    subset='training',
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_ROOT,
    validation_split=VALIDATION_SPLIT,
    subset='validation',
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

class_names = train_ds.class_names
num_classes = len(class_names)
print('Classes:', num_classes)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000, seed=SEED).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

## 4. Class Weights\n
\n
Class weights help when medicinal plant classes have fewer images than the old crop classes.

In [ ]:
total = sum(counts.values())
class_weight = {}
for idx, name in enumerate(class_names):
    class_weight[idx] = total / (num_classes * max(1, counts.get(name, 1)))

print('Sample class weights:')
for idx in range(min(10, num_classes)):
    print(idx, class_names[idx], round(class_weight[idx], 3))

## 5. Build Model With ECA Attention

EfficientNetV2 includes preprocessing internally when `include_preprocessing=True`. For this notebook, deploy with `MODEL_FAMILY=none` because preprocessing is saved inside the model.

This model adds an **Efficient Channel Attention (ECA)** block after the CNN backbone. ECA is lightweight and helps the classifier focus on disease-specific channels such as spots, mildew texture, rust color, and leaf damage patterns. Keep it only if validation accuracy and confusion matrix improve compared with the baseline.


In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.12),
    layers.RandomContrast(0.12),
], name='augmentation')

def eca_block(inputs, gamma=2, b=1, name='eca'):
    """Efficient Channel Attention block for CNN feature maps."""
    channels = inputs.shape[-1]
    kernel_size = int(abs((np.log2(float(channels)) + b) / gamma))
    kernel_size = kernel_size if kernel_size % 2 else kernel_size + 1
    kernel_size = max(3, kernel_size)

    x = layers.GlobalAveragePooling2D(name=f'{name}_gap')(inputs)
    x = layers.Reshape((-1, 1), name=f'{name}_reshape_in')(x)
    x = layers.Conv1D(1, kernel_size=kernel_size, padding='same', use_bias=False, name=f'{name}_conv1d')(x)
    x = layers.Activation('sigmoid', name=f'{name}_sigmoid')(x)
    x = layers.Reshape((1, 1, channels), name=f'{name}_reshape_out')(x)
    return layers.Multiply(name=f'{name}_scale')([inputs, x])

if MODEL_NAME == 'efficientnetv2b3':
    base_model = keras.applications.EfficientNetV2B3(
        include_top=False, weights='imagenet', input_shape=(IMG_SIZE, IMG_SIZE, 3), include_preprocessing=True
    )
else:
    base_model = keras.applications.EfficientNetV2B0(
        include_top=False, weights='imagenet', input_shape=(IMG_SIZE, IMG_SIZE, 3), include_preprocessing=True
    )

base_model.trainable = False

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = eca_block(x, name='disease_eca')
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.35)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)
model = keras.Model(inputs, outputs, name=f'AgroVision_{MODEL_NAME}_ECA_combined')

model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_accuracy')]
)

model.summary()


## 6. Train Classifier Head

In [ ]:
checkpoint_path = OUTPUT_DIR / f'{MODEL_NAME}_combined_best.keras'

callbacks = [
    keras.callbacks.ModelCheckpoint(checkpoint_path, monitor='val_accuracy', save_best_only=True, verbose=1),
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.35, patience=2, min_lr=1e-6, verbose=1),
]

history_head = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=INITIAL_EPOCHS,
    class_weight=class_weight,
    callbacks=callbacks
)

## 7. Fine-Tune Model\n
\n
Fine-tuning the last layers improves accuracy on plant disease textures. If overfitting starts, reduce `FINE_TUNE_LAYERS` or epochs.

In [ ]:
FINE_TUNE_LAYERS = 45
base_model.trainable = True

for layer in base_model.layers[:-FINE_TUNE_LAYERS]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=2e-5, weight_decay=1e-6),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_accuracy')]
)

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=INITIAL_EPOCHS + FINE_TUNE_EPOCHS,
    initial_epoch=len(history_head.history['loss']),
    class_weight=class_weight,
    callbacks=callbacks
)

## 8. Evaluate

In [ ]:
results = model.evaluate(val_ds, verbose=1)
print(dict(zip(model.metrics_names, results)))

## 9. Save `.h5` Model And Class Names\n
\n
These files can be copied into the project `models/` folder.

In [ ]:
h5_path = OUTPUT_DIR / f'{MODEL_NAME}_combined.h5'
class_path = OUTPUT_DIR / 'class_names_combined.json'

model.save(h5_path)
with open(class_path, 'w', encoding='utf-8') as f:
    json.dump({'class_names': class_names}, f, indent=2)

print('Saved model:', h5_path)
print('Saved classes:', class_path)
print('Class count:', len(class_names))

## 10. Test Single Image\n
\n
Upload or point to a test leaf image.

In [ ]:
from google.colab import files
from PIL import Image
import matplotlib.pyplot as plt

uploaded = files.upload()
test_path = next(iter(uploaded.keys()))

img = keras.utils.load_img(test_path, target_size=(IMG_SIZE, IMG_SIZE))
arr = keras.utils.img_to_array(img)
batch = np.expand_dims(arr, axis=0)
pred = model.predict(batch, verbose=0)[0]
top_idx = pred.argsort()[-3:][::-1]

plt.imshow(Image.open(test_path))
plt.axis('off')
plt.show()

for idx in top_idx:
    print(class_names[idx], float(pred[idx]))

## 11. Use In AgroVision Backend\n
\n
Copy outputs into the local project:\n
\n
```text\n
models/EfficientNetV2_combined.h5\n
models/class_names_combined.json\n
```\n
\n
Then set backend `.env`:\n
\n
```env\n
MODEL_PATH=../models/efficientnetv2b0_combined.h5\n
CLASS_NAMES_PATH=../models/class_names_combined.json\n
MODEL_FAMILY=none\n
IMAGE_SIZE=224\n
```\n
\n
`MODEL_FAMILY=none` is recommended here because EfficientNetV2 was saved with preprocessing inside the model.